# 07 figures

Regenerate report PNG/SVG figures from saved stacks and acquisition tables. Shared RGB/index scales, dated acquisition spans, NoData legends, map scales and matching histogram CSVs are explicit. No retrieval or processing rerun is needed.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yaml').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT.parent / 'work' / 'mplconfig'))
from pipeline.config import load_config
from pipeline.forest_notebooks import preview
from IPython.display import display
cfg = load_config(ROOT / 'config.yaml')
SENSORS = list(cfg['forest_change']['enabled_sensors'])
# Edit config.yaml first. An in-session override may instead be made here.
# SENSORS = ['landsat', 'sentinel2', 'opera', 'hyp3']
OUT = cfg.path(cfg['forest_change']['outputs'])
print('Enabled sensors:', SENSORS)


In [ ]:
from pipeline.forest_stack import load_stack
from pipeline.forest_figures import optical_figures, temporal_figures
from pipeline.forest_sar_figures import sar_figures
import pandas as pd
for sensor in SENSORS:
    ds = load_stack(cfg,sensor)
    if sensor in ['landsat','sentinel2']:
        paths = optical_figures(cfg,ds)
        if (OUT/f'{sensor}_per_acquisition.csv').exists(): paths.append(temporal_figures(cfg,sensor))
    else: paths = sar_figures(cfg,ds,pd.read_csv(OUT/f'{sensor}_profiles.csv'))
    for path in paths: preview(path)
if 'sentinel2' in SENSORS:
    from pipeline.forest_figures import context_map
    preview(context_map(cfg,load_stack(cfg,'sentinel2')))